In [ ]:
# Install required packages (Kaggle)
%pip install -q ultralytics pyyaml pillow matplotlib

# YOLOv8n on PASCAL VOC 2007 (Kaggle)


In [2]:
# Detect dataset location (Kaggle or local workspace)
from pathlib import Path
# Edit these paths to match your local or Kaggle dataset locations.
TRAINVAL_ROOT = Path(r'D:/100DaysofML/Notebooks/099_YOLO/VOCtrainval_06-Nov-2007/VOCdevkit/VOC2007')
TEST_ROOT = Path(r'D:/100DaysofML/Notebooks/099_YOLO/VOCtest_06-Nov-2007/VOCdevkit/VOC2007')
TRAINVAL_JPEG_DIR = TRAINVAL_ROOT / 'JPEGImages'
TRAINVAL_ANN_DIR = TRAINVAL_ROOT / 'Annotations'
TRAINVAL_IMAGESETS_MAIN = TRAINVAL_ROOT / 'ImageSets' / 'Main'
TEST_JPEG_DIR = TEST_ROOT / 'JPEGImages'
TEST_ANN_DIR = TEST_ROOT / 'Annotations'
TEST_IMAGESETS_MAIN = TEST_ROOT / 'ImageSets' / 'Main'
WORK_ROOT = Path(r'D:/100DaysofML/Notebooks/099_YOLO/voc2007_yolo')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
LABELS_DIR = WORK_ROOT / 'labels'
LABELS_DIR.mkdir(parents=True, exist_ok=True)
print('Using TRAINVAL_ROOT =', TRAINVAL_ROOT)
print('Using TEST_ROOT =', TEST_ROOT)
print('Using WORK_ROOT =', WORK_ROOT)

Using TRAINVAL_ROOT = D:\100DaysofML\Notebooks\099_YOLO\VOCtrainval_06-Nov-2007\VOCdevkit\VOC2007
Using TEST_ROOT = D:\100DaysofML\Notebooks\099_YOLO\VOCtest_06-Nov-2007\VOCdevkit\VOC2007
Using WORK_ROOT = D:\100DaysofML\Notebooks\099_YOLO\voc2007_yolo


In [3]:
# Classes and conversion helper
CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor'
]
CLASS_MAP = {c: i for i, c in enumerate(CLASSES)}
import xml.etree.ElementTree as ET
def convert_xml_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    yolo_lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in CLASS_MAP:
            continue
        cls = CLASS_MAP[name]
        bnd = obj.find('bndbox')
        xmin = float(bnd.find('xmin').text)
        ymin = float(bnd.find('ymin').text)
        xmax = float(bnd.find('xmax').text)
        ymax = float(bnd.find('ymax').text)
        x_center = (xmin + xmax) / 2.0 / img_w
        y_center = (ymin + ymax) / 2.0 / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h
        yolo_lines.append(f'{cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')
    return yolo_lines

In [4]:
# Build splits and convert annotations
from sklearn.model_selection import train_test_split
from PIL import Image
trainval_images = sorted(TRAINVAL_JPEG_DIR.glob('*.jpg')) + sorted(TRAINVAL_JPEG_DIR.glob('*.jpeg')) + sorted(TRAINVAL_JPEG_DIR.glob('*.png'))
test_images = sorted(TEST_JPEG_DIR.glob('*.jpg')) + sorted(TEST_JPEG_DIR.glob('*.jpeg')) + sorted(TEST_JPEG_DIR.glob('*.png'))
if not trainval_images:
    raise FileNotFoundError(f'No images found in TRAINVAL_JPEG_DIR: {TRAINVAL_JPEG_DIR}')
if not test_images:
    raise FileNotFoundError(f'No images found in TEST_JPEG_DIR: {TEST_JPEG_DIR}')
train_images, val_images = train_test_split(trainval_images, test_size=0.2, random_state=42, shuffle=True)
train_names = [p.stem for p in train_images]
val_names = [p.stem for p in val_images]
test_names = [p.stem for p in test_images]
count = 0
for xml in (TRAINVAL_ANN_DIR).glob('*.xml'):
    stem = xml.stem
    img_path = next((candidate for candidate in [
        TRAINVAL_JPEG_DIR / f'{stem}.jpg',
        TRAINVAL_JPEG_DIR / f'{stem}.jpeg',
        TRAINVAL_JPEG_DIR / f'{stem}.png',
    ] if candidate.exists()), None)
    if img_path is None:
        continue
    tree = ET.parse(xml)
    root = tree.getroot()
    size = root.find('size')
    if size is not None:
        w = float(size.find('width').text)
        h = float(size.find('height').text)
    else:
        w, h = Image.open(img_path).size
    yolo_lines = convert_xml_to_yolo(xml, w, h)
    if yolo_lines:
        out_path = LABELS_DIR / f'{stem}.txt'
        out_path.write_text('\n'.join(yolo_lines), encoding='utf-8')
        count += 1
print('Wrote', count, 'label files to', LABELS_DIR)
print('Train images:', len(train_names))
print('Val images:', len(val_names))
print('Test images:', len(test_names))

FileNotFoundError: No images found in TRAINVAL_JPEG_DIR: D:\100DaysofML\Notebooks\099_YOLO\VOCtrainval_06-Nov-2007\VOCdevkit\VOC2007\JPEGImages

In [ ]:
# Write train/val/test text files and dataset.yaml
def abs_paths_for(names):
    return [str((TRAINVAL_JPEG_DIR / (n if n.endswith('.jpg') else n + '.jpg')).resolve()) for n in names]
train_list = abs_paths_for(train_names)
val_list = abs_paths_for(val_names)
test_list = [str((TEST_JPEG_DIR / (n if n.endswith('.jpg') else n + '.jpg')).resolve()) for n in test_names]
missing_train = [p for p in train_list if not Path(p).exists()]
missing_val = [p for p in val_list if not Path(p).exists()]
missing_test = [p for p in test_list if not Path(p).exists()]
if missing_train or missing_val or missing_test:
    raise FileNotFoundError(f'Missing image paths. train={len(missing_train)} val={len(missing_val)} test={len(missing_test)}')
if not train_list or not val_list or not test_list:
    raise ValueError(f'Empty split detected. train={len(train_list)} val={len(val_list)} test={len(test_list)}')
(WORK_ROOT / 'train.txt').write_text('\n'.join(train_list), encoding='utf-8')
(WORK_ROOT / 'val.txt').write_text('\n'.join(val_list), encoding='utf-8')
(WORK_ROOT / 'test.txt').write_text('\n'.join(test_list), encoding='utf-8')
dataset_yaml = {
    'names': CLASSES,
    'nc': len(CLASSES),
    'train': str((WORK_ROOT / 'train.txt').resolve()),
    'val': str((WORK_ROOT / 'val.txt').resolve()),
    'test': str((WORK_ROOT / 'test.txt').resolve())
}
import yaml
with open(WORK_ROOT / 'dataset.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_yaml, f)
print('Wrote dataset.yaml to', (WORK_ROOT / 'dataset.yaml').resolve())

## Train on Kaggle GPU

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
dataset_yaml_path = WORK_ROOT / 'dataset.yaml'
if not dataset_yaml_path.exists():
    raise FileNotFoundError(f'Missing dataset.yaml at {dataset_yaml_path}. Run the label conversion and dataset export cells first.')
dataset_yaml = str(dataset_yaml_path.resolve())
model.train(data=dataset_yaml, epochs=50, imgsz=640, batch=16, project='/kaggle/working/runs', name='yolov8n_voc2007')

In [ ]:
metrics = model.val(data=dataset_yaml, split="val")

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

In [5]:
test_metrics = model.val(data=dataset_yaml, split="test")

print("Test precision:", test_metrics.box.mp)
print("Test recall:", test_metrics.box.mr)
print("Test mAP@0.5:", test_metrics.box.map50)
print("Test mAP@0.5:0.95:", test_metrics.box.map)

NameError: name 'model' is not defined